In [0]:

from pyspark.sql import functions as F

# 1. Read the permanent Bronze orders table
bronze_orders = spark.table(
    "workspace.ecom_poc.bronze_orders"
)

# 2. Standardize text and safely convert data types
orders_typed = (
    bronze_orders

    # Clean text values
    .withColumn(
        "order_id_clean",
        F.trim(F.col("order_id"))
    )
    .withColumn(
        "customer_id_clean",
        F.trim(F.col("customer_id"))
    )
    .withColumn(
        "category_clean",
        F.initcap(F.trim(F.col("category")))
    )
    .withColumn(
        "country_clean",
        F.trim(F.col("country"))
    )
    .withColumn(
        "order_status_clean",
        F.upper(F.trim(F.col("order_status")))
    )

    # Safe data type conversions
    .withColumn(
        "quantity_typed",
        F.expr("try_cast(trim(quantity) AS INT)")
    )
    .withColumn(
        "unit_price_typed",
        F.expr("try_cast(trim(unit_price) AS DECIMAL(18,2))")
    )
    .withColumn(
        "order_date_typed",
        F.expr("try_cast(trim(order_date) AS DATE)")
    )
)

print("Type conversion prepared successfully!")

orders_typed.printSchema()


In [0]:

# Find records needing further validation or normalization

problem_examples = orders_typed.filter(
    F.col("quantity_typed").isNull()
    | (F.col("quantity_typed") <= 0)
    | F.col("unit_price_typed").isNull()
    | (F.col("unit_price_typed") < 0)
    | F.col("order_date_typed").isNull()
    | (F.col("order_status") != F.col("order_status_clean"))
)

display(
    problem_examples.select(
        "order_id",
        "quantity",
        "quantity_typed",
        "unit_price",
        "unit_price_typed",
        "order_date",
        "order_date_typed",
        "order_status",
        "order_status_clean"
    ).limit(15)
)


In [0]:

from pyspark.sql import functions as F

# Fixed reference date for reproducible validation
as_of_date = F.lit("2026-11-19").cast("date")

# Define allowed values
allowed_statuses = [
    "COMPLETED", "CANCELLED", "PENDING", "SHIPPED"
]

allowed_categories = [
    "Electronics", "Clothing", "Books", "Home"
]

allowed_countries = [
    "Bangladesh", "India", "USA", "UK"
]

# Apply all order validation rules
orders_validated = orders_typed.withColumn(
    "rejection_reason",
    F.concat_ws(
        " | ",

        # Rule 1: Order ID must exist
        F.when(
            F.col("order_id_clean").isNull()
            | (F.col("order_id_clean") == ""),
            F.lit("MISSING_ORDER_ID")
        ),

        # Rule 2: Customer ID must exist
        F.when(
            F.col("customer_id_clean").isNull()
            | (F.col("customer_id_clean") == ""),
            F.lit("MISSING_CUSTOMER_ID")
        ),

        # Rule 3: Quantity must be valid
        F.when(
            F.col("quantity_typed").isNull()
            | (F.col("quantity_typed") <= 0)
            | (F.col("quantity_typed") > 100),
            F.lit("INVALID_QUANTITY")
        ),

        # Rule 4: Price must be valid
        F.when(
            F.col("unit_price_typed").isNull()
            | (F.col("unit_price_typed") <= 0)
            | (F.col("unit_price_typed") > 100000),
            F.lit("INVALID_PRICE")
        ),

        # Rule 5: Date must be valid and non-future
        F.when(
            F.col("order_date_typed").isNull()
            | (F.col("order_date_typed") > as_of_date),
            F.lit("INVALID_DATE")
        ),

        # Rule 6: Allowed order status
        F.when(
            F.col("order_status_clean").isNull()
            | ~F.col("order_status_clean").isin(
                *allowed_statuses
            ),
            F.lit("INVALID_STATUS")
        ),

        # Rule 7: Allowed product category
        F.when(
            F.col("category_clean").isNull()
            | ~F.col("category_clean").isin(
                *allowed_categories
            ),
            F.lit("INVALID_CATEGORY")
        ),

        # Rule 8: Allowed country
        F.when(
            F.col("country_clean").isNull()
            | ~F.col("country_clean").isin(
                *allowed_countries
            ),
            F.lit("INVALID_COUNTRY")
        )
    )
)

print("Order validation rules applied!")


In [0]:

# Count validation outcomes
summary = orders_validated.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(
            F.col("rejection_reason") == "", 1
        ).otherwise(0)
    ).alias("valid_rows"),

    F.sum(
        F.when(
            F.col("rejection_reason") != "", 1
        ).otherwise(0)
    ).alias("invalid_rows")
).first()

print("Total Bronze records:", summary["total_rows"])
print("Valid records:", summary["valid_rows"])
print("Invalid records:", summary["invalid_rows"])

assert summary["total_rows"] == 1_000_000

assert (
    summary["valid_rows"] + summary["invalid_rows"]
    == summary["total_rows"]
)

print("VALIDATION COUNT CHECK: PASSED")


In [0]:

display(
    orders_validated
    .filter(F.col("rejection_reason") != "")
    .select(
        "order_id",
        "quantity",
        "unit_price",
        "order_date",
        "rejection_reason"
    )
    .limit(20)
)


In [0]:

from pyspark.sql import functions as F

# 1. Select invalid orders only
quarantine_orders = orders_validated.filter(
    F.col("rejection_reason") != ""
)

# 2. Save invalid orders as a Delta table
(
    quarantine_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.ecom_poc.quarantine_orders")
)

# 3. Verify saved records
saved_quarantine = spark.table(
    "workspace.ecom_poc.quarantine_orders"
)

total_quarantined = saved_quarantine.count()

print("Quarantined orders:", total_quarantined)

assert total_quarantined == 5666

print("QUARANTINE CREATION: PASSED")


In [0]:

# Count records by rejection reason

reason_counts = (
    saved_quarantine
    .select(
        F.explode(
            F.split(
                F.col("rejection_reason"),
                r"\s*\|\s*"
            )
        ).alias("reason")
    )
    .groupBy("reason")
    .count()
    .orderBy(F.desc("count"))
)

display(reason_counts)


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Keep only records that passed validation
valid_orders = orders_validated.filter(
    F.col("rejection_reason") == ""
)

# Define how to select one winner per order_id
order_window = (
    Window
    .partitionBy("order_id_clean")
    .orderBy(
        F.col("quantity_typed").asc(),
        F.col("_row_hash").asc(),
        F.col("_source_row_id").asc()
    )
)

# Rank records within each order_id group
ranked_orders = (
    valid_orders
    .withColumn(
        "_dedupe_rank",
        F.row_number().over(order_window)
    )
    .withColumn(
        "_winner_hash",
        F.first("_row_hash").over(order_window)
    )
)

print("Order deduplication ranking prepared!")


In [0]:

# 1. Select winning records
silver_orders = (
    ranked_orders
    .filter(F.col("_dedupe_rank") == 1)
    .select(
        F.col("order_id_clean").alias("order_id"),
        F.col("customer_id_clean").alias("customer_id"),
        F.col("order_date_typed").alias("order_date"),
        F.col("category_clean").alias("category"),
        F.col("quantity_typed").alias("quantity"),
        F.col("unit_price_typed").alias("unit_price"),
        F.col("order_status_clean").alias("order_status"),
        F.col("country_clean").alias("country"),
        "_source_row_id",
        "_row_hash",
        "_batch_id",
        "_source_file",
        "_ingested_at"
    )
)

# 2. Select duplicate losers
duplicate_orders = (
    ranked_orders
    .filter(F.col("_dedupe_rank") > 1)
    .withColumn(
        "duplicate_reason",
        F.when(
            F.col("_row_hash") == F.col("_winner_hash"),
            F.lit("EXACT_DUPLICATE")
        ).otherwise(
            F.lit("DUPLICATE_KEY_DIFFERENT_VALUES")
        )
    )
)

# 3. Save accepted Silver orders
(
    silver_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.ecom_poc.silver_orders")
)

# 4. Save duplicate losers
(
    duplicate_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.ecom_poc.duplicate_orders")
)

print("Silver orders table created!")
print("Duplicate orders table created!")


In [0]:

silver_count = spark.table(
    "workspace.ecom_poc.silver_orders"
).count()

duplicate_count = spark.table(
    "workspace.ecom_poc.duplicate_orders"
).count()

quarantine_count = spark.table(
    "workspace.ecom_poc.quarantine_orders"
).count()

print("Silver orders:", silver_count)
print("Duplicate losers:", duplicate_count)
print("Quarantined orders:", quarantine_count)

assert silver_count == 984_082
assert duplicate_count == 10_252
assert quarantine_count == 5_666

print("COUNT VERIFICATION: PASSED")

# Examine duplicate types
display(
    spark.table("workspace.ecom_poc.duplicate_orders")
    .groupBy("duplicate_reason")
    .count()
)


In [0]:

from pyspark.sql import functions as F

# 1. Get original Bronze record IDs
bronze_ids = (
    spark.table("workspace.ecom_poc.bronze_orders")
    .select("_source_row_id")
)

# 2. Get accepted Silver record IDs
silver_ids = (
    spark.table("workspace.ecom_poc.silver_orders")
    .select("_source_row_id")
    .withColumn("outcome", F.lit("SILVER"))
)

# 3. Get quarantined record IDs
quarantine_ids = (
    spark.table("workspace.ecom_poc.quarantine_orders")
    .select("_source_row_id")
    .withColumn("outcome", F.lit("QUARANTINE"))
)

# 4. Get duplicate loser record IDs
duplicate_ids = (
    spark.table("workspace.ecom_poc.duplicate_orders")
    .select("_source_row_id")
    .withColumn("outcome", F.lit("DUPLICATE"))
)

# 5. Combine all three outcomes
all_outcomes = (
    silver_ids
    .unionByName(quarantine_ids)
    .unionByName(duplicate_ids)
)

# 6. Count Bronze and outcome records
bronze_count = bronze_ids.count()
outcome_count = all_outcomes.count()

# 7. Find records assigned more than once
multiple_outcomes = (
    all_outcomes
    .groupBy("_source_row_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 8. Find Bronze records missing from all outcomes
missing_records = (
    bronze_ids
    .join(
        all_outcomes.select("_source_row_id"),
        on="_source_row_id",
        how="left_anti"
    )
    .count()
)

# 9. Find outcome IDs that do not exist in Bronze
unknown_records = (
    all_outcomes
    .select("_source_row_id")
    .join(
        bronze_ids,
        on="_source_row_id",
        how="left_anti"
    )
    .count()
)

# 10. Verify no outcome has a missing ID
null_ids = all_outcomes.filter(
    F.col("_source_row_id").isNull()
).count()

# 11. Display the verification results
print("Bronze records:", bronze_count)
print("Outcome records:", outcome_count)
print("Multiple outcomes:", multiple_outcomes)
print("Missing records:", missing_records)
print("Unknown records:", unknown_records)
print("Null outcome IDs:", null_ids)

# 12. Enforce the required conditions
assert bronze_count == 1_000_000
assert outcome_count == bronze_count
assert multiple_outcomes == 0
assert missing_records == 0
assert unknown_records == 0
assert null_ids == 0

print("EXACT ORDER RECONCILIATION: PASSED")


In [0]:

from pyspark.sql import functions as F

# Read the original Bronze transactions
bronze_transactions = spark.table(
    "workspace.ecom_poc.bronze_transactions"
)

# Standardize text and convert data types
transactions_typed = (
    bronze_transactions

    # Clean identifiers
    .withColumn(
        "txn_id_clean",
        F.trim(F.col("txn_id"))
    )
    .withColumn(
        "order_id_clean",
        F.trim(F.col("order_id"))
    )

    # Standardize payment information
    .withColumn(
        "payment_method_clean",
        F.upper(F.trim(F.col("payment_method")))
    )
    .withColumn(
        "currency_clean",
        F.upper(F.trim(F.col("currency")))
    )
    .withColumn(
        "txn_status_clean",
        F.upper(F.trim(F.col("txn_status")))
    )

    # Convert transaction amount
    .withColumn(
        "amount_typed",
        F.expr("try_cast(trim(amount) AS DECIMAL(18,2))")
    )

    # Convert transaction timestamp
    .withColumn(
        "txn_ts_typed",
        F.expr("try_cast(trim(txn_ts) AS TIMESTAMP)")
    )
)

print("Transaction type conversion prepared!")

transactions_typed.printSchema()


In [0]:

# Find interesting transactions for inspection

reference_time = F.lit(
    "2026-11-19 23:59:59"
).cast("timestamp")

problem_transactions = transactions_typed.filter(
    F.col("txn_id_clean").isNull()
    | (F.col("txn_id_clean") == "")
    | F.col("order_id_clean").isNull()
    | (F.col("order_id_clean") == "")
    | F.col("amount_typed").isNull()
    | (F.col("amount_typed") <= 0)
    | F.col("txn_ts_typed").isNull()
    | (F.col("txn_ts_typed") > reference_time)
    | (F.col("currency_clean") == "XYZ")
    | (F.col("txn_status") != F.col("txn_status_clean"))
)

display(
    problem_transactions.select(
        "txn_id",
        "order_id",
        "amount",
        "amount_typed",
        "currency",
        "txn_status",
        "txn_status_clean",
        "txn_ts",
        "txn_ts_typed"
    ).limit(20)
)


In [0]:

from pyspark.sql import functions as F

# Fixed validation reference time
reference_time = F.to_timestamp(
    F.lit("2026-11-19 23:59:59")
)

# Allowed values
allowed_currencies = ["BDT", "INR", "USD", "GBP"]

allowed_methods = [
    "CARD",
    "BANK_TRANSFER",
    "MOBILE_BANKING",
    "CASH_ON_DELIVERY"
]

allowed_statuses = [
    "SUCCESS",
    "FAILED",
    "PENDING",
    "REFUNDED"
]

# 1. Get accepted order IDs for reference validation
valid_order_ids = (
    spark.table("workspace.ecom_poc.silver_orders")
    .select(
        F.col("order_id").alias("_matched_order_id")
    )
)

# 2. Match transactions against accepted orders
transactions_joined = transactions_typed.join(
    valid_order_ids,
    transactions_typed["order_id_clean"]
    == valid_order_ids["_matched_order_id"],
    "left"
)

# 3. Apply validation rules
transactions_validated = transactions_joined.withColumn(
    "rejection_reason",
    F.concat_ws(
        " | ",

        # Missing transaction ID
        F.when(
            F.col("txn_id_clean").isNull()
            | (F.col("txn_id_clean") == ""),
            F.lit("MISSING_TXN_ID")
        ),

        # Missing order ID
        F.when(
            F.col("order_id_clean").isNull()
            | (F.col("order_id_clean") == ""),
            F.lit("MISSING_ORDER_ID")
        ),

        # Invalid payment amount
        F.when(
            F.col("amount_typed").isNull()
            | (F.col("amount_typed") <= 0)
            | (F.col("amount_typed") > 1000000),
            F.lit("INVALID_AMOUNT")
        ),

        # Unsupported currency
        F.when(
            F.col("currency_clean").isNull()
            | ~F.col("currency_clean").isin(
                *allowed_currencies
            ),
            F.lit("INVALID_CURRENCY")
        ),

        # Unsupported payment method
        F.when(
            F.col("payment_method_clean").isNull()
            | ~F.col("payment_method_clean").isin(
                *allowed_methods
            ),
            F.lit("INVALID_PAYMENT_METHOD")
        ),

        # Unsupported payment status
        F.when(
            F.col("txn_status_clean").isNull()
            | ~F.col("txn_status_clean").isin(
                *allowed_statuses
            ),
            F.lit("INVALID_TXN_STATUS")
        ),

        # Invalid or future transaction timestamp
        F.when(
            F.col("txn_ts_typed").isNull()
            | (F.col("txn_ts_typed") > reference_time),
            F.lit("INVALID_TIMESTAMP")
        ),

        # Orphan order reference
        F.when(
            F.col("order_id_clean").isNotNull()
            & (F.col("order_id_clean") != "")
            & F.col("_matched_order_id").isNull(),
            F.lit("ORPHAN_ORDER_ID")
        )
    )
)

print("Transaction validation rules applied!")


In [0]:

# Calculate validation summary

summary = transactions_validated.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(
            F.col("rejection_reason") == "", 1
        ).otherwise(0)
    ).alias("valid_rows"),

    F.sum(
        F.when(
            F.col("rejection_reason") != "", 1
        ).otherwise(0)
    ).alias("invalid_rows")
).first()

print("Total transactions:", summary["total_rows"])
print("Valid transactions:", summary["valid_rows"])
print("Invalid transactions:", summary["invalid_rows"])

assert summary["total_rows"] == 1_000_000

assert (
    summary["valid_rows"] + summary["invalid_rows"]
    == summary["total_rows"]
)

print("TRANSACTION VALIDATION CHECK: PASSED")

# Count failures by rule
reason_counts = (
    transactions_validated
    .filter(F.col("rejection_reason") != "")
    .select(
        F.explode(
            F.split(
                F.col("rejection_reason"),
                r"\s*\|\s*"
            )
        ).alias("reason")
    )
    .groupBy("reason")
    .count()
    .orderBy(F.desc("count"))
)

display(reason_counts)


In [0]:

from pyspark.sql import functions as F

# 1. Select invalid transactions
quarantine_transactions = (
    transactions_validated
    .filter(F.col("rejection_reason") != "")
)

# 2. Save as a permanent Delta table
(
    quarantine_transactions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.ecom_poc.quarantine_transactions"
    )
)

# 3. Verify the saved records
saved_quarantine = spark.table(
    "workspace.ecom_poc.quarantine_transactions"
)

quarantine_count = saved_quarantine.count()

print("Quarantined transactions:", quarantine_count)

assert quarantine_count == 8317

print("TRANSACTION QUARANTINE: PASSED")


In [0]:

# Count each rejection reason from the saved table

reason_counts = (
    saved_quarantine
    .select(
        F.explode(
            F.split(
                F.col("rejection_reason"),
                r"\s*\|\s*"
            )
        ).alias("reason")
    )
    .groupBy("reason")
    .count()
    .orderBy(F.desc("count"))
)

display(reason_counts)


In [0]:

from pyspark.sql import functions as F

audit_check = saved_quarantine.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("_source_row_id").alias("unique_ids"),
    F.sum(
        F.when(
            F.col("_source_row_id").isNull(), 1
        ).otherwise(0)
    ).alias("null_ids")
).first()

print("Quarantine rows:", audit_check["total_rows"])
print("Unique source IDs:", audit_check["unique_ids"])
print("Null source IDs:", audit_check["null_ids"])

assert audit_check["total_rows"] == 8317
assert audit_check["unique_ids"] == 8317
assert audit_check["null_ids"] == 0

print("QUARANTINE AUDIT CHECK: PASSED")


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Keep only transactions that passed validation
valid_transactions = transactions_validated.filter(
    F.col("rejection_reason") == ""
)

# 2. Count identical content within each transaction ID
content_counts = (
    valid_transactions
    .groupBy("txn_id_clean", "_row_hash")
    .agg(
        F.count("*").alias("_content_occurrences")
    )
)

# 3. Attach the content frequency to each record
transactions_with_counts = valid_transactions.join(
    content_counts,
    on=["txn_id_clean", "_row_hash"],
    how="left"
)

# 4. Define the deterministic winner-selection rule
transaction_window = (
    Window
    .partitionBy("txn_id_clean")
    .orderBy(
        F.col("_content_occurrences").desc(),
        F.col("_row_hash").asc(),
        F.col("_source_row_id").asc()
    )
)

# 5. Assign ranks and identify the winning hash
ranked_transactions = (
    transactions_with_counts
    .withColumn(
        "_dedupe_rank",
        F.row_number().over(transaction_window)
    )
    .withColumn(
        "_winner_hash",
        F.first("_row_hash").over(
            transaction_window.rowsBetween(
                Window.unboundedPreceding,
                Window.unboundedFollowing
            )
        )
    )
)

print("Transaction deduplication ranking prepared!")


In [0]:

# Separate winners and duplicate losers logically
transaction_winners = ranked_transactions.filter(
    F.col("_dedupe_rank") == 1
)

transaction_losers = (
    ranked_transactions
    .filter(F.col("_dedupe_rank") > 1)
    .withColumn(
        "duplicate_reason",
        F.when(
            F.col("_row_hash") == F.col("_winner_hash"),
            F.lit("EXACT_DUPLICATE")
        ).otherwise(
            F.lit("DUPLICATE_KEY_DIFFERENT_VALUES")
        )
    )
)

# Verify counts before writing any tables
winner_count = transaction_winners.count()
loser_count = transaction_losers.count()

print("Transaction winners:", winner_count)
print("Duplicate losers:", loser_count)

assert winner_count + loser_count == 991_683

print("DEDUPLICATION COUNT CHECK: PASSED")

# Inspect loser categories
display(
    transaction_losers
    .groupBy("duplicate_reason")
    .count()
)


In [0]:

from pyspark.sql import functions as F

# 1. Prepare the accepted Silver transactions
silver_transactions = transaction_winners.select(
    F.col("txn_id_clean").alias("txn_id"),
    F.col("order_id_clean").alias("order_id"),
    F.col("payment_method_clean").alias("payment_method"),
    F.col("amount_typed").alias("amount"),
    F.col("currency_clean").alias("currency"),
    F.col("txn_status_clean").alias("txn_status"),
    F.col("txn_ts_typed").alias("txn_ts"),

    # Preserve Bronze audit metadata
    "_source_row_id",
    "_row_hash",
    "_batch_id",
    "_source_file",
    "_ingested_at"
)

# 2. Prepare duplicate losers
# Keep original fields, audit metadata and duplicate reasons
duplicate_transactions = transaction_losers

# 3. Save accepted transactions
(
    silver_transactions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.ecom_poc.silver_transactions"
    )
)

print("Silver transactions table created!")

# 4. Save duplicate losers
(
    duplicate_transactions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.ecom_poc.duplicate_transactions"
    )
)

print("Duplicate transactions table created!")


In [0]:

from pyspark.sql import functions as F

# Read saved Delta tables
silver_df = spark.table(
    "workspace.ecom_poc.silver_transactions"
)

duplicate_df = spark.table(
    "workspace.ecom_poc.duplicate_transactions"
)

quarantine_df = spark.table(
    "workspace.ecom_poc.quarantine_transactions"
)

# Count each outcome
silver_count = silver_df.count()
duplicate_count = duplicate_df.count()
quarantine_count = quarantine_df.count()

# Verify transaction business-key uniqueness
unique_txn_ids = silver_df.select(
    "txn_id"
).distinct().count()

print("Silver transactions:", silver_count)
print("Duplicate losers:", duplicate_count)
print("Quarantined transactions:", quarantine_count)
print("Unique Silver transaction IDs:", unique_txn_ids)

# Check expected results
assert silver_count == 980_082
assert duplicate_count == 11_601
assert quarantine_count == 8_317
assert unique_txn_ids == silver_count

# Verify total reconciliation counts
assert (
    silver_count + duplicate_count + quarantine_count
    == 1_000_000
)

print("TRANSACTION COUNT VERIFICATION: PASSED")


In [0]:

from pyspark.sql import functions as F

# 1. Original Bronze transaction identifiers
bronze_ids = (
    spark.table("workspace.ecom_poc.bronze_transactions")
    .select("_source_row_id")
)

# 2. Accepted Silver transaction identifiers
silver_ids = (
    spark.table("workspace.ecom_poc.silver_transactions")
    .select("_source_row_id")
    .withColumn("outcome", F.lit("SILVER"))
)

# 3. Quarantined transaction identifiers
quarantine_ids = (
    spark.table("workspace.ecom_poc.quarantine_transactions")
    .select("_source_row_id")
    .withColumn("outcome", F.lit("QUARANTINE"))
)

# 4. Duplicate loser identifiers
duplicate_ids = (
    spark.table("workspace.ecom_poc.duplicate_transactions")
    .select("_source_row_id")
    .withColumn("outcome", F.lit("DUPLICATE"))
)

# 5. Combine all three outcomes
all_outcomes = (
    silver_ids
    .unionByName(quarantine_ids)
    .unionByName(duplicate_ids)
)

# 6. Count original and processed records
bronze_count = bronze_ids.count()
outcome_count = all_outcomes.count()

# 7. Detect multiple assignments
multiple_outcomes = (
    all_outcomes
    .groupBy("_source_row_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 8. Find Bronze records missing from outcomes
missing_records = (
    bronze_ids
    .join(
        all_outcomes.select("_source_row_id"),
        on="_source_row_id",
        how="left_anti"
    )
    .count()
)

# 9. Find outcome records not present in Bronze
unknown_records = (
    all_outcomes
    .select("_source_row_id")
    .join(
        bronze_ids,
        on="_source_row_id",
        how="left_anti"
    )
    .count()
)

# 10. Check missing identifiers
null_ids = all_outcomes.filter(
    F.col("_source_row_id").isNull()
).count()

# 11. Print results
print("Bronze transactions:", bronze_count)
print("Total outcome records:", outcome_count)
print("Multiple outcomes:", multiple_outcomes)
print("Missing records:", missing_records)
print("Unknown records:", unknown_records)
print("Null source IDs:", null_ids)

# 12. Verify reconciliation
assert bronze_count == 1_000_000
assert outcome_count == bronze_count
assert multiple_outcomes == 0
assert missing_records == 0
assert unknown_records == 0
assert null_ids == 0

print("EXACT TRANSACTION RECONCILIATION: PASSED")
